# Experiments using Le World Model

## Preparation

Convert config and weights to checkpoint. Also renames the model layer names to fix a compatibility error of the state dict given in the pushT repo example.

Note that `weights_old.pt` contains the pushT weights that are given in the repo.

In [ ]:
import json
import os
from pathlib import Path

import stable_worldmodel as swm
import torch
from hydra.utils import instantiate

from jepa import JEPA

os.environ["STABLEWM_HOME"] = Path.cwd().as_posix()
src = Path(swm.data.utils.get_cache_dir(sub_folder="checkpoints"), "pushT")
out = Path(swm.data.utils.get_cache_dir(sub_folder="checkpoints"), "pushT", "weights.pt")
out_ckpt = Path(swm.data.utils.get_cache_dir(sub_folder="checkpoints"), "pushT", "lewm_object.ckpt")

cfg = json.loads((src / "config.json").read_text())

model = JEPA(
    encoder=instantiate(cfg["encoder"]),
    predictor=instantiate(cfg["predictor"]),
    action_encoder=instantiate(cfg["action_encoder"]),
    projector=instantiate(cfg["projector"]),
    pred_proj=instantiate(cfg["pred_proj"]),
)

state_dict = torch.load(src / "weights_old.pt", map_location="cpu", weights_only=False)
translated_state_dict = {}
for key, tensor in state_dict.items():
    new_key = key

    # Target the mismatched ViT layers
    if "encoder.encoder.layer." in key:
        # Fix the base layer prefix
        new_key = new_key.replace("encoder.encoder.layer.", "encoder.layers.")

        # Translate Attention Q, K, V Projections
        new_key = new_key.replace("attention.attention.query", "attention.q_proj")
        new_key = new_key.replace("attention.attention.key", "attention.k_proj")
        new_key = new_key.replace("attention.attention.value", "attention.v_proj")

        # Translate Attention Output
        new_key = new_key.replace("attention.output.dense", "attention.o_proj")

        # Translate MLP / FeedForward layers
        new_key = new_key.replace("intermediate.dense", "mlp.fc1")
        new_key = new_key.replace("output.dense", "mlp.fc2")

    translated_state_dict[new_key] = tensor

model.load_state_dict(translated_state_dict, strict=True)
out.parent.mkdir(parents=True, exist_ok=True)
torch.save(translated_state_dict, out)
torch.save(model, out_ckpt)

17:02:54 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}


Folder Structure needs to be:

```
{{STABLEWM_HOME}}/
├── checkpoints/                # is hardcoded
│   └── some_path/              # optional
│       ├── weights.pt          # named as specified in policy path below
│       └── config.json         # is hardcoded
└── datasets/                   # is hardcoded
    └── pusht_expert_train.h5   # named as specified in config/eval/pusht.yaml
```

In [ ]:
os.environ["STABLEWM_HOME"] = Path.cwd().as_posix()

## Evaluation

The policy path has to point to the weights.pt, relative to the `checkpoints/` folder, i.e. pushT/weights.pt

In [ ]:
!python eval.py --config-name=pusht.yaml policy=pushT/weights.pt

## Training

In [ ]:
!python train.py data=pusht

## Explore the dataset

In [ ]:
import h5py

dataset_path = "./datasets/pusht_expert_train.h5"

with h5py.File(dataset_path, "r") as f:
    def explore(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[Group] {name}")
        else:
            print(f"[Dataset] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(explore)

In [9]:
import numpy as np

dataset_path = "../viz_sa/blinc_sa_crossing_s05_a07_ep0000_speed_agility.npz"
data = np.load(dataset_path)

for key in data.files:
    arr = data[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")


agent_ids: shape=(5,), dtype=int64
range_speed_min: shape=(), dtype=float64
range_speed_max: shape=(), dtype=float64
range_accel_min: shape=(), dtype=float64
range_accel_max: shape=(), dtype=float64
range_tau_min: shape=(), dtype=float64
range_tau_max: shape=(), dtype=float64
a1_t: shape=(801,), dtype=float64
a1_xy: shape=(801, 2), dtype=float64
a1_vxy: shape=(801, 2), dtype=float64
a1_spd: shape=(801,), dtype=float64
a1_t_a: shape=(800,), dtype=float64
a1_amag: shape=(800,), dtype=float64
a1_hrate: shape=(800,), dtype=float64
a1_frames: shape=(801,), dtype=int64
a1_speed_proxy: shape=(801,), dtype=float64
a1_agility_proxy: shape=(800,), dtype=float64
a1_pred_t: shape=(793,), dtype=float64
a1_v_hat: shape=(793,), dtype=float64
a1_a_hat: shape=(793,), dtype=float64
a1_v_true: shape=(793,), dtype=float64
a1_a_true: shape=(793,), dtype=float64
a1_label_t: shape=(397,), dtype=float64
a1_label_speed: shape=(397,), dtype=float64
a1_label_agility: shape=(397,), dtype=float64
a1_label_v0: shap

Convert the .npz file into .h5 format compatible with the world model training

In [14]:
import re

import h5py
import numpy as np

npz_path = "../viz_sa/blinc_sa_crossing_s00_a00_ep0000_speed_agility.npz"
h5_path = "../viz_sa/blinc_sa_crossing_s00_a00_ep0000_speed_agility.h5"
keymap = {
    "pixels": "frames",         # shape: [T, H, W, C]
    "action": "vxy",            # shape: [T, A]
    "proprio": ["xy", "vxy"],   # shape: [T, P]
    "state": ["xy", "vxy"],     # shape: [T, S]
}

data = np.load(npz_path)

with h5py.File(h5_path, "w") as h5f:
    for h5_name, patterns in keymap.items():

        if isinstance(patterns, str):
            patterns = [patterns]

        pattern_cols = []

        for pattern in patterns:
            regex = r"a\d+_" + pattern
            matched_keys = [k for k in data if re.fullmatch(regex, k)]

            if len(matched_keys) == 0:
                raise KeyError(f"No keys in {npz_path} match pattern '{regex}'")

            arrays = [data[k] for k in matched_keys]
            pattern_col = np.concatenate(arrays, axis=0)
            pattern_cols.append(pattern_col)
            print(f"{h5_name}: Matched keys from '{pattern}': {matched_keys}")

        concat = np.column_stack(pattern_cols)
        h5f.create_dataset(h5_name, data=concat)

        print(f"Created dataset '{h5_name}', Shape: {concat.shape}, Dtype: {concat.dtype}")

print(f"Converted {npz_path} → {h5_path}")

pixels: Matched keys from 'frames': ['a1_frames', 'a2_frames', 'a3_frames', 'a4_frames', 'a5_frames']
Created dataset 'pixels', Shape: (4005, 1), Dtype: int64
action: Matched keys from 'vxy': ['a1_vxy', 'a2_vxy', 'a3_vxy', 'a4_vxy', 'a5_vxy']
Created dataset 'action', Shape: (4005, 2), Dtype: float64
proprio: Matched keys from 'xy': ['a1_xy', 'a2_xy', 'a3_xy', 'a4_xy', 'a5_xy']
proprio: Matched keys from 'vxy': ['a1_vxy', 'a2_vxy', 'a3_vxy', 'a4_vxy', 'a5_vxy']
Created dataset 'proprio', Shape: (4005, 4), Dtype: float64
state: Matched keys from 'xy': ['a1_xy', 'a2_xy', 'a3_xy', 'a4_xy', 'a5_xy']
state: Matched keys from 'vxy': ['a1_vxy', 'a2_vxy', 'a3_vxy', 'a4_vxy', 'a5_vxy']
Created dataset 'state', Shape: (4005, 4), Dtype: float64
Converted ../viz_sa/blinc_sa_crossing_s00_a00_ep0000_speed_agility.npz → ../viz_sa/blinc_sa_crossing_s00_a00_ep0000_speed_agility.h5


In [ ]:
data = np.load(npz_path)

# Identify all episode numbers (a1_, a2_, ...)
episode_numbers = sorted({
    int(re.match(r"a(\d+)_", k).group(1))
    for k in data.keys()
    if re.match(r"a\d+_", k)
})

# Compute episode lengths from any one pattern (e.g., first keymap entry)
first_pattern = keymap[next(iter(keymap))]
if isinstance(first_pattern, list):
    first_pattern = first_pattern[0]

ep_lens = []
for ep in episode_numbers:
    regex = rf"a{ep}_" + first_pattern
    keys = [k for k in data.keys() if re.fullmatch(regex, k)]
    if len(keys) == 0:
        raise KeyError(f"No keys for episode {ep} with pattern {first_pattern}")
    ep_lens.append(data[keys[0]].shape[0])

# Compute offsets
ep_offsets = np.cumsum([0] + ep_lens[:-1])

# Build episode_idx and step_idx
episode_idx = np.concatenate([
    np.full(ep_lens[i], episode_numbers[i], dtype=np.int32)
    for i in range(len(ep_lens))
])

step_idx = np.concatenate([
    np.arange(ep_lens[i], dtype=np.int32)
    for i in range(len(ep_lens))
])

In [ ]:
import os
import re

import h5py
import numpy as np


def npz_folder_to_h5(folder, h5_path, keymap):
    """
    Convert all NPZ files in a folder into one H5 dataset.
    Episode numbering continues across files.
    """

    # Collect all NPZ files
    npz_files = sorted([f for f in os.listdir(folder) if f.endswith(".npz")])

    # Global containers
    all_vertical = {name: [] for name in keymap}
    global_episode_idx = []
    global_step_idx = []
    global_ep_len = []
    global_ep_offset = []

    next_episode_number = 1
    current_offset = 0

    for npz_file in npz_files:
        path = os.path.join(folder, npz_file)
        data = np.load(path)

        # Identify episodes in this file
        episode_numbers_local = sorted({
            int(re.match(r"a(\d+)_", k).group(1))
            for k in data
            if re.match(r"a\d+_", k)
        })

        # Determine episode lengths using first pattern
        first_pattern = keymap[next(iter(keymap))]
        if isinstance(first_pattern, list):
            first_pattern = first_pattern[0]

        ep_lens_local = []
        for ep in episode_numbers_local:
            regex = rf"a{ep}_" + first_pattern
            keys = [k for k in data if re.fullmatch(regex, k)]
            ep_lens_local.append(data[keys[0]].shape[0])

        # Build episode_idx and step_idx for this file
        for i, ep_len in enumerate(ep_lens_local):
            ep_global = next_episode_number
            global_episode_idx.append(np.full(ep_len, ep_global, dtype=np.int32))
            global_step_idx.append(np.arange(ep_len, dtype=np.int32))
            global_ep_len.append(ep_len)
            global_ep_offset.append(current_offset)

            current_offset += ep_len
            next_episode_number += 1

        # Process all datasets in keymap
        for h5_name, patterns in keymap.items():

            if isinstance(patterns, str):
                patterns = [patterns]

            vertical_segments = []

            for pattern in patterns:
                regex = r"a\d+_" + pattern
                matched_keys = sorted([k for k in data
                                       if re.fullmatch(regex, k)])

                arrays = [data[k] for k in matched_keys]
                vertical_concat = np.concatenate(arrays, axis=0)
                vertical_segments.append(vertical_concat)

            if len(vertical_segments) == 1:
                final = vertical_segments[0]
            else:
                final = np.concatenate(vertical_segments, axis=1)

            all_vertical[h5_name].append(final)

    # Concatenate across files
    with h5py.File(h5_path, "w") as h5f:

        # Episode metadata
        h5f.create_dataset("episode_idx", data=np.concatenate(global_episode_idx))
        h5f.create_dataset("step_idx", data=np.concatenate(global_step_idx))
        h5f.create_dataset("ep_len", data=np.array(global_ep_len, dtype=np.int32))
        h5f.create_dataset("ep_offset", data=np.array(global_ep_offset, dtype=np.int32))

        # Main datasets
        for h5_name, segments in all_vertical.items():
            h5f.create_dataset(h5_name, data=np.concatenate(segments, axis=0))

    print(f"Converted folder '{folder}' → {h5_path}")
